# Домашнее задание №1

## Представление изображений, бинаризация, шумы, фильтрация, морфология

### Цель

Освоить базовые операции над растровым изображением и научиться сравнивать методы предобработки количественно, а не визуально. Результатом работы является не «красивая» отфильтрованная картинка, а обоснованный вывод о том, какой фильтр применим к какому типу шума и при каких условиях бинаризация с последующей морфологической очисткой даёт устойчивый результат.

[Методические указания блока](README.md) · [Общие МУ](../../../docs/guidelines-students.md) · [Рубрика оценивания](../teachers-assessment/README.md)

## 1. Что используется в работе

Программный стек (других библиотек в работе не требуется):

| Библиотека | Роль в работе |
|---|---|
| `opencv-python` (`cv2`) | цветовые преобразования, фильтрация, пороги, морфология |
| `numpy` | работа с массивами, модели шума, БПФ |
| `scikit-image` | тестовые изображения, метрики PSNR и SSIM |
| `matplotlib` | визуализация |
| `pandas` | сводные таблицы журнала экспериментов |

Данные: встроенная коллекция `skimage.data` (`camera`, `coins`, `page`, `text`, `astronaut`) и синтетические изображения, генерируемые кодом ноутбука. Интернет не требуется. Карточка набора — в [реестре датасетов блока](../../resources/datasets/README.md).

По условию задания набор должен содержать **не менее 5 изображений, включая цветные и полутоновые**. Заготовка формирует такой набор; вы можете добавить в него собственные снимки, указав источник.

Перед сдачей заполните шапку работы: ФИО, группа, номер работы, версии библиотек, seed (требование п. 1 [общих МУ](../../../docs/guidelines-students.md)).

In [ ]:
# Служебная ячейка: импорты, версии, seed.
import time
from dataclasses import dataclass, asdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import skimage
from skimage import data as skdata
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

SEED = 42
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

AUTHOR = {"fio": "", "group": "", "work": "ДЗ1"}   # TODO: заполните

VERSIONS = {
    "opencv": cv2.__version__,
    "numpy": np.__version__,
    "scikit-image": skimage.__version__,
    "pandas": pd.__version__,
    "seed": SEED,
}
VERSIONS

## 2. Краткая теоретическая справка

### 2.1. Представление изображения

Полутоновое изображение — матрица $I \in \mathbb{Z}^{H \times W}$, обычно `uint8` с диапазоном $[0, 255]$. Цветное изображение — тензор $H \times W \times 3$. OpenCV читает и хранит цветные изображения в порядке каналов **BGR**, `skimage` и `matplotlib` — в **RGB**. Несогласованность порядка каналов — самая частая причина «странных» цветов на визуализациях.

Цветовые пространства:

- **RGB** — аддитивная модель устройства вывода; каналы сильно коррелированы, яркость «размазана» по всем трём.
- **HSV** — тон $H$, насыщенность $S$, яркость $V$; удобно для пороговой сегментации по цвету, так как отделяет цветность от освещённости.
- **Lab** — $L$ — светлота, $a, b$ — оппонентные цветовые координаты; приближённо перцептуально равномерно, евклидово расстояние в Lab примерно соответствует воспринимаемому различию цветов.

### 2.2. Модели шума

Аддитивный гауссов шум:

$$ I_{\text{noisy}}(x,y) = I(x,y) + n(x,y), \qquad n \sim \mathcal{N}(0, \sigma^2) $$

Импульсный шум «соль-перец»: с вероятностью $p/2$ пиксель заменяется на 0, с вероятностью $p/2$ — на 255. Это не аддитивная модель: исходное значение пикселя теряется полностью.

### 2.3. Фильтрация

Линейная фильтрация — свёртка с ядром $K$:

$$ (I * K)(x,y) = \sum_{i=-k}^{k} \sum_{j=-k}^{k} K(i,j)\, I(x-i, y-j) $$

Усредняющее ядро $K = \frac{1}{n^2}\mathbf{1}_{n \times n}$; гауссово ядро

$$ K_\sigma(i,j) = \frac{1}{2\pi\sigma^2} \exp\!\left(-\frac{i^2 + j^2}{2\sigma^2}\right) $$

Медианный фильтр нелинеен: значение пикселя заменяется медианой окрестности. Медиана — робастная статистика, единичные выбросы не смещают её, тогда как среднее смещается пропорционально амплитуде выброса.

### 2.4. Частотная фильтрация

Теорема о свёртке: свёртке в пространственной области соответствует поэлементное произведение спектров:

$$ \mathcal{F}\{I * K\} = \mathcal{F}\{I\} \cdot \mathcal{F}\{K\} $$

Поэтому фильтрацию можно выполнять маскированием спектра $\mathcal{F}\{I\}$: низкочастотная маска сглаживает, высокочастотная выделяет границы. Идеальная (прямоугольная) маска в частотной области даёт бесконечный по протяжённости фильтр в пространственной, что проявляется как звон (эффект Гиббса).

### 2.5. Метрики качества восстановления

$$ \mathrm{MSE} = \frac{1}{HW}\sum_{x,y} \bigl(I(x,y) - \hat I(x,y)\bigr)^2, \qquad
\mathrm{PSNR} = 10 \log_{10} \frac{L^2}{\mathrm{MSE}} $$

где $L$ — максимальное значение шкалы (255 для `uint8`).

$$ \mathrm{SSIM}(x,y) = \frac{(2\mu_x\mu_y + C_1)(2\sigma_{xy} + C_2)}{(\mu_x^2 + \mu_y^2 + C_1)(\sigma_x^2 + \sigma_y^2 + C_2)} $$

**Обе метрики считаются между результатом фильтрации и эталоном (чистым изображением), а не между результатом и зашумлённым входом.** Сравнение с зашумлённым входом измеряет степень «похожести на шум» и является типичной ошибкой, снижающей оценку.

### 2.6. Бинаризация

Глобальный порог: $B(x,y) = [\,I(x,y) > t\,]$. Метод Оцу выбирает $t$, максимизируя межклассовую дисперсию:

$$ t^{*} = \arg\max_t \; \omega_0(t)\,\omega_1(t)\,\bigl(\mu_0(t) - \mu_1(t)\bigr)^2 $$

Оцу предполагает бимодальную гистограмму и единое освещение всего кадра. При неравномерном освещении используется адаптивный порог, вычисляемый в окне: $t(x,y) = \mathrm{mean}(W_{x,y}) - C$ или взвешенное гауссовым ядром среднее.

### 2.7. Морфология

Для бинарного изображения $B$ и структурного элемента $S$:

$$ (B \ominus S)(x) = \bigwedge_{s \in S} B(x+s), \qquad (B \oplus S)(x) = \bigvee_{s \in S} B(x+s) $$

Открытие $B \circ S = (B \ominus S) \oplus S$ удаляет мелкие светлые объекты (одиночную «соль») и разрывает тонкие перемычки. Закрытие $B \bullet S = (B \oplus S) \ominus S$ заполняет мелкие тёмные дыры («перец») и соединяет близкие компоненты.

## 3. Задачи

Формулировка по [методическим указаниям блока](README.md).

Для выбранного набора изображений (не менее 5, включая цветные и полутоновые):

1. Исследуйте цветовые пространства (RGB, HSV, Lab) и постройте гистограммы каналов.
2. Добавьте шум двух типов (гауссов, «соль-перец») и сравните не менее трёх методов фильтрации (усредняющая, медианная, гауссова; дополнительно — частотная фильтрация через БПФ).
3. Выполните бинаризацию (глобальный порог, Оцу, адаптивная) и очистите результат морфологическими операциями (эрозия, дилатация, открытие, закрытие).

**Ожидаемый результат:** таблица «тип шума × фильтр → PSNR/SSIM», визуальные серии «до/после», вывод о применимости методов.

**Что будет проверяться** ([рубрика](../teachers-assessment/README.md)): не менее 3 фильтров и 2 типов шума; PSNR/SSIM посчитаны по эталону; Оцу и адаптивный порог сравнены на неравномерно освещённом изображении; морфологические операции применены осмысленно (открытие против «соли», закрытие против «перца»).

## 4. Данные

Набор формируется служебной функцией ниже. В него входят полутоновые изображения (`camera`, `coins`, `page`, `text`), цветное (`astronaut`) и синтетическая страница с неравномерным освещением — последняя нужна для честного сравнения Оцу и адаптивного порога.

Все изображения приводятся к `uint8`. Полутоновые хранятся как $H \times W$, цветные — как $H \times W \times 3$ в порядке **RGB**.

In [ ]:
# Служебная ячейка: формирование набора данных. Изменять не требуется.

def make_uneven_page(height: int = 384, width: int = 512, seed: int = SEED) -> np.ndarray:
    '''Синтетическая «страница» с текстовыми штрихами и неравномерным освещением.

    Возвращает: uint8 [H, W] — полутоновое изображение,
    а также используется как эталон для проверки бинаризации через make_uneven_page_gt().
    '''
    rng = np.random.default_rng(seed)
    page = np.full((height, width), 235, dtype=np.uint8)
    for row in range(40, height - 40, 26):
        x = 40
        while x < width - 60:
            w = int(rng.integers(18, 70))
            if x + w > width - 40:
                break
            cv2.rectangle(page, (x, row), (x + w, row + 9), 35, -1)
            x += w + int(rng.integers(8, 20))
    yy, xx = np.mgrid[0:height, 0:width]
    gradient = 1.0 - 0.55 * ((xx / width) ** 1.4) - 0.25 * (yy / height)
    lit = np.clip(page.astype(np.float64) * gradient + 25, 0, 255)
    return lit.astype(np.uint8)


def make_uneven_page_gt(height: int = 384, width: int = 512, seed: int = SEED) -> np.ndarray:
    '''Эталонная бинарная маска штрихов для make_uneven_page (True = штрих).'''
    rng = np.random.default_rng(seed)
    mask = np.zeros((height, width), dtype=np.uint8)
    for row in range(40, height - 40, 26):
        x = 40
        while x < width - 60:
            w = int(rng.integers(18, 70))
            if x + w > width - 40:
                break
            cv2.rectangle(mask, (x, row), (x + w, row + 9), 1, -1)
            x += w + int(rng.integers(8, 20))
    return mask.astype(bool)


def to_uint8(image: np.ndarray) -> np.ndarray:
    '''Привести изображение любого типа к uint8 [0, 255].'''
    if image.dtype == np.uint8:
        return image
    if image.dtype == bool:
        return (image.astype(np.uint8) * 255)
    arr = image.astype(np.float64)
    if arr.max() <= 1.0:
        arr = arr * 255.0
    return np.clip(arr, 0, 255).astype(np.uint8)


def load_dataset() -> dict:
    '''Набор изображений работы: имя -> uint8 массив (grayscale [H,W] или RGB [H,W,3]).'''
    return {
        "camera": to_uint8(skdata.camera()),
        "coins": to_uint8(skdata.coins()),
        "page": to_uint8(skdata.page()),
        "text": to_uint8(skdata.text()),
        "astronaut": to_uint8(skdata.astronaut()),      # цветное, RGB
        "uneven_page": make_uneven_page(),              # синтетика, неравномерный свет
    }


IMAGES = load_dataset()
pd.DataFrame(
    [{"name": k, "shape": v.shape, "dtype": str(v.dtype),
      "kind": "color" if v.ndim == 3 else "gray",
      "min": int(v.min()), "max": int(v.max())} for k, v in IMAGES.items()]
)

In [ ]:
# Служебная ячейка: визуализация. Изменять не требуется.

def show_row(images, titles=None, figsize_scale: float = 3.2, cmap: str = "gray") -> None:
    '''Показать список изображений в один ряд с подписями.'''
    n = len(images)
    titles = titles or [""] * n
    fig, axes = plt.subplots(1, n, figsize=(figsize_scale * n, figsize_scale))
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        if img.ndim == 3:
            ax.imshow(img)
        else:
            ax.imshow(img, cmap=cmap, vmin=0, vmax=255)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_hist(channels, labels, title: str = "") -> None:
    '''Гистограммы каналов на одних осях (256 бинов, диапазон [0, 255]).'''
    plt.figure(figsize=(6, 3))
    for ch, label in zip(channels, labels):
        hist = cv2.calcHist([ch.astype(np.uint8)], [0], None, [256], [0, 256]).ravel()
        plt.plot(hist, label=label, linewidth=1)
    plt.title(title, fontsize=10)
    plt.xlabel("значение")
    plt.ylabel("число пикселей")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


show_row([IMAGES["camera"], IMAGES["coins"], IMAGES["astronaut"], IMAGES["uneven_page"]],
         ["camera", "coins", "astronaut (RGB)", "uneven_page"])

In [ ]:
# Служебная ячейка: журнал экспериментов. Изменять не требуется.
# Каждый запуск конфигурации записывается одной строкой; сводные таблицы строятся из журнала.

RUNS: list = []


def log_run(**fields) -> dict:
    '''Записать один эксперимент в журнал.

    Обязательные поля по методике курса: image, stage, method, params (dict), метрики.
    Возвращает записанную строку.
    '''
    row = {"seed": SEED, **fields}
    if isinstance(row.get("params"), dict):
        row["params"] = ", ".join(f"{k}={v}" for k, v in row["params"].items())
    RUNS.append(row)
    return row


def runs_table(stage: str = None) -> pd.DataFrame:
    '''Журнал экспериментов в виде таблицы (опционально — только один этап).'''
    df = pd.DataFrame(RUNS)
    if stage is not None and not df.empty:
        df = df[df["stage"] == stage]
    return df.reset_index(drop=True)


def timed(func, *args, **kwargs):
    '''Выполнить функцию, вернуть (результат, время в миллисекундах).'''
    start = time.perf_counter()
    result = func(*args, **kwargs)
    return result, (time.perf_counter() - start) * 1e3


print("Журнал инициализирован, записей:", len(RUNS))

## 5. Задание 1. Цветовые пространства и гистограммы каналов

Ниже — рабочий пример на одном изображении: разложение `astronaut` по RGB, HSV и Lab с гистограммами каналов. Он задаёт формат ожидаемого результата. Ваша задача — распространить его на весь набор и сделать содержательные наблюдения (какое пространство отделяет объект от фона, какой канал устойчив к изменению освещённости).

Обратите внимание: `cv2.cvtColor` для 8-битных изображений масштабирует $H$ в $[0, 179]$, а $a$ и $b$ в Lab — в $[0, 255]$ со сдвигом 128. Это нужно учитывать при интерпретации гистограмм.

In [ ]:
# Рабочий пример (рельсы): одно изображение, три цветовых пространства.
rgb = IMAGES["astronaut"]
hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)

show_row([rgb, hsv[:, :, 0], hsv[:, :, 1], lab[:, :, 0]],
         ["RGB", "HSV: H", "HSV: S", "Lab: L"])

show_hist([rgb[:, :, 0], rgb[:, :, 1], rgb[:, :, 2]], ["R", "G", "B"], "astronaut, RGB")
show_hist([hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]], ["H", "S", "V"], "astronaut, HSV")

In [ ]:
# TODO (задание 1.1): исследуйте цветовые пространства на всём наборе.
#
# Требования:
# 1. Для каждого цветного изображения набора постройте разложение в RGB, HSV и Lab
#    и гистограммы всех каналов (используйте show_row и show_hist).
# 2. Для полутоновых изображений постройте гистограмму яркости и опишите её форму
#    (число мод, наличие фоновой моды) — это понадобится в разделе бинаризации.
# 3. Добавьте не менее двух собственных цветных изображений или синтезируйте
#    цветные сцены кодом, если хотите расширить набор.
# 4. Зафиксируйте наблюдения в разделе «Отчёт»: какое пространство и какой канал
#    вы выбрали бы для отделения объекта от фона и почему.

def analyze_color_spaces(image: np.ndarray, name: str) -> dict:
    '''Разложить изображение по RGB/HSV/Lab, показать каналы и гистограммы.

    Вход:
        image : np.ndarray, uint8, [H, W] или [H, W, 3] в порядке RGB
        name  : str, идентификатор изображения для подписей и журнала
    Выход:
        dict с описательными статистиками каналов (среднее, стандартное отклонение,
        энтропия или доля насыщенных пикселей) — по одному ключу на канал.
        Дополнительно функция выводит визуализации.
    '''
    raise NotImplementedError


# for name, img in IMAGES.items():
#     analyze_color_spaces(img, name)

## 6. Задание 2. Модели шума

Служебные функции ниже реализуют две обязательные модели шума. Они детерминированы при фиксированном `seed`, поэтому серии воспроизводимы.

Ключевое различие: гауссов шум сохраняет исходное значение пикселя (добавка с нулевым средним), «соль-перец» его уничтожает. Именно поэтому усреднение работает для первого и не работает для второго.

In [ ]:
# Служебная ячейка: модели шума. Изменять не требуется.

def add_gaussian_noise(image: np.ndarray, sigma: float = 20.0, seed: int = SEED) -> np.ndarray:
    '''Аддитивный гауссов шум с нулевым средним.

    Вход: uint8 изображение, sigma в единицах шкалы [0, 255].
    Выход: uint8 изображение той же формы.
    '''
    rng = np.random.default_rng(seed)
    noisy = image.astype(np.float64) + rng.normal(0.0, sigma, image.shape)
    return np.clip(noisy, 0, 255).astype(np.uint8)


def add_salt_pepper(image: np.ndarray, amount: float = 0.05, seed: int = SEED) -> np.ndarray:
    '''Импульсный шум «соль-перец»: доля amount пикселей заменяется на 0 или 255.'''
    rng = np.random.default_rng(seed)
    noisy = image.copy()
    mask = rng.random(image.shape[:2])
    salt = mask < amount / 2
    pepper = (mask >= amount / 2) & (mask < amount)
    if noisy.ndim == 3:
        noisy[salt] = 255
        noisy[pepper] = 0
    else:
        noisy[salt] = 255
        noisy[pepper] = 0
    return noisy


NOISE_MODELS = {
    "gaussian_s20": lambda img: add_gaussian_noise(img, sigma=20.0),
    "salt_pepper_005": lambda img: add_salt_pepper(img, amount=0.05),
}

clean = IMAGES["camera"]
show_row([clean, NOISE_MODELS["gaussian_s20"](clean), NOISE_MODELS["salt_pepper_005"](clean)],
         ["эталон", "гауссов, sigma=20", "соль-перец, p=0.05"])

## 7. Задание 2. Фильтрация и количественное сравнение

Метрики PSNR и SSIM считаются **между результатом фильтрации и эталоном** — чистым изображением до внесения шума. Эталон в этой работе известен, поскольку шум вносится искусственно; это и делает количественное сравнение возможным.

Заранее зафиксируйте baseline: строка «без фильтрации» (то есть метрики зашумлённого изображения относительно эталона). Без неё нельзя утверждать, что фильтр что-то улучшил.

Рабочий пример ниже выполняет один полный проход: изображение → шум → два фильтра → метрики → запись в журнал.

In [ ]:
# Служебная ячейка: метрики. Изменять не требуется.

def psnr(reference: np.ndarray, test: np.ndarray) -> float:
    '''PSNR между эталоном и результатом, дБ. Оба аргумента uint8.'''
    return float(peak_signal_noise_ratio(reference, test, data_range=255))


def ssim(reference: np.ndarray, test: np.ndarray) -> float:
    '''SSIM между эталоном и результатом. Для цветных изображений — по каналам.'''
    if reference.ndim == 3:
        return float(structural_similarity(reference, test, data_range=255, channel_axis=2))
    return float(structural_similarity(reference, test, data_range=255))


# Рабочий пример (рельсы): baseline + два фильтра на одном изображении и одном шуме.
reference = IMAGES["camera"]
noisy = NOISE_MODELS["salt_pepper_005"](reference)

log_run(image="camera", stage="filtering", noise="salt_pepper_005", method="none",
        params={}, psnr=psnr(reference, noisy), ssim=ssim(reference, noisy), ms=0.0)

for method, func in [("mean_3", lambda x: cv2.blur(x, (3, 3))),
                     ("median_3", lambda x: cv2.medianBlur(x, 3))]:
    result, ms = timed(func, noisy)
    log_run(image="camera", stage="filtering", noise="salt_pepper_005", method=method,
            params={"ksize": 3}, psnr=psnr(reference, result), ssim=ssim(reference, result), ms=ms)

show_row([reference, noisy, cv2.blur(noisy, (3, 3)), cv2.medianBlur(noisy, 3)],
         ["эталон", "соль-перец", "усредняющий 3x3", "медианный 3x3"])
runs_table("filtering")

In [ ]:
# TODO (задание 2.1): реализуйте единый диспетчер фильтров.
#
# Обязательный минимум: усредняющий, медианный, гауссов. Дополнительно можно
# добавить билатеральный (cv2.bilateralFilter) и частотный (раздел 8).

def apply_filter(image: np.ndarray, method: str, **params) -> np.ndarray:
    '''Применить фильтр к изображению.

    Вход:
        image  : np.ndarray, uint8, [H, W] или [H, W, 3]
        method : str, один из {"mean", "median", "gaussian", ...}
        params : параметры фильтра (ksize, sigma и т. п.)
    Выход:
        np.ndarray, uint8, той же формы и типа, что вход.
    Требования:
        - размер ядра медианного фильтра должен быть нечётным;
        - функция не должна менять входной массив на месте;
        - неизвестный method должен приводить к явной ошибке, а не к молчаливому
          возврату исходного изображения.
    '''
    raise NotImplementedError

In [ ]:
# TODO (задание 2.2): проведите контролируемую серию.
#
# Серия: {не менее 5 изображений} x {2 типа шума} x {не менее 3 фильтров + baseline "none"}
# x {не менее 2 размеров ядра}. В каждой строке журнала должны быть:
# image, noise, method, params, psnr, ssim, ms.
#
# Требования методики (см. общие МУ, п. 2):
# - в одной серии изменяется один фактор; размер ядра и тип фильтра не варьируйте
#   одновременно с уровнем шума в рамках одного сравнения;
# - baseline "none" обязателен для каждой пары (изображение, шум);
# - метрики считаются относительно эталона, а не относительно зашумлённого входа.
#
# Подсказка по структуре кода:
# for name, ref in IMAGES.items():
#     for noise_name, noise_fn in NOISE_MODELS.items():
#         noisy = noise_fn(ref)
#         log_run(... method="none" ...)
#         for method, params in FILTER_GRID:
#             result, ms = timed(apply_filter, noisy, method, **params)
#             log_run(...)

FILTER_GRID = [
    # ("mean", {"ksize": 3}), ("mean", {"ksize": 5}),
    # ("median", {"ksize": 3}), ("median", {"ksize": 5}),
    # ("gaussian", {"ksize": 5, "sigma": 1.0}),
]

# TODO: цикл серии

runs_table("filtering")

## 8. Задание 2 (дополнительно). Частотная фильтрация через БПФ

Ячейка ниже показывает спектр изображения и порядок работы: `fft2` → `fftshift` → маска → `ifftshift` → `ifft2` → действительная часть. Ваша задача — реализовать сами маски и включить частотный фильтр в общую серию наравне с пространственными.

Обратите внимание на артефакт звона у идеальной (резкой) маски и сравните его с гауссовой маской того же радиуса среза. Этот эффект — прямое следствие теоремы о свёртке (раздел 2.4) и хороший материал для защиты.

In [ ]:
# Рабочий пример (рельсы): спектр изображения и его визуализация.
gray = IMAGES["camera"]
spectrum = np.fft.fftshift(np.fft.fft2(gray.astype(np.float64)))
log_magnitude = np.log1p(np.abs(spectrum))
log_magnitude_vis = to_uint8(log_magnitude / log_magnitude.max())

show_row([gray, log_magnitude_vis], ["изображение", "лог-амплитуда спектра"])
print("Форма спектра:", spectrum.shape, "| центр (нулевая частота):",
      spectrum.shape[0] // 2, spectrum.shape[1] // 2)

In [ ]:
# TODO (задание 2.3): частотная фильтрация.

def frequency_mask(shape: tuple, cutoff: float, kind: str = "lowpass",
                   profile: str = "ideal") -> np.ndarray:
    '''Построить маску в частотной области (после fftshift, ноль частот в центре).

    Вход:
        shape   : (H, W)
        cutoff  : радиус среза в пикселях спектра
        kind    : "lowpass" | "highpass"
        profile : "ideal" (резкая граница) | "gaussian" (плавная)
    Выход:
        np.ndarray, float64, [H, W], значения в [0, 1].
    '''
    raise NotImplementedError


def frequency_filter(image: np.ndarray, cutoff: float, kind: str = "lowpass",
                     profile: str = "ideal") -> np.ndarray:
    '''Отфильтровать полутоновое изображение маскированием спектра.

    Выход: uint8 [H, W]. Последовательность: fft2 -> fftshift -> маска ->
    ifftshift -> ifft2 -> np.real -> обрезка в [0, 255].
    '''
    raise NotImplementedError


# TODO: сравните ideal и gaussian маски при одинаковом cutoff, покажите звон,
# добавьте не менее двух частотных конфигураций в журнал через log_run
# со stage="filtering", чтобы они попали в общую сводную таблицу.

## 9. Задание 3. Бинаризация

Сравнение проводится на изображениях с равномерным и неравномерным освещением. На `uneven_page` известна эталонная маска штрихов (`make_uneven_page_gt`), поэтому качество бинаризации можно оценить количественно — по F1 на пикселях, а не «на глаз». Для реальных изображений (`page`, `text`) оценка визуальная, и это ограничение нужно указать в выводах.

Типичная ошибка: показать Оцу и адаптивный порог только на равномерно освещённом кадре. На таком материале разницы почти нет, и вывод о преимуществе адаптивного порога оказывается ничем не подкреплён.

In [ ]:
# Служебная ячейка: пиксельные метрики бинарных масок + рабочий пример.

def pixel_prf(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    '''Precision / recall / F1 для бинарных масок (True = объект).'''
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)
    tp = float(np.logical_and(y_true, y_pred).sum())
    fp = float(np.logical_and(~y_true, y_pred).sum())
    fn = float(np.logical_and(y_true, ~y_pred).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}


# Рабочий пример (рельсы): глобальный порог и Оцу на синтетической странице.
page = IMAGES["uneven_page"]
page_gt = make_uneven_page_gt()

_, global_bin = cv2.threshold(page, 127, 255, cv2.THRESH_BINARY_INV)
otsu_t, otsu_bin = cv2.threshold(page, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
print("Порог, выбранный методом Оцу:", otsu_t)

for method, binary in [("global_127", global_bin), ("otsu", otsu_bin)]:
    log_run(image="uneven_page", stage="binarization", method=method,
            params={"invert": True}, **pixel_prf(page_gt, binary > 0))

show_row([page, global_bin, otsu_bin, to_uint8(page_gt)],
         ["uneven_page", "глобальный t=127", f"Оцу t={int(otsu_t)}", "эталон"])
runs_table("binarization")

In [ ]:
# TODO (задание 3.1): полное сравнение методов бинаризации.
#
# Требуется: глобальный порог (не менее двух значений t), Оцу, адаптивный
# (cv2.adaptiveThreshold, режимы MEAN_C и GAUSSIAN_C, не менее двух размеров блока).
# Обязательно включите в сравнение изображение с неравномерным освещением.

def binarize(image: np.ndarray, method: str, **params) -> np.ndarray:
    '''Бинаризовать полутоновое изображение.

    Вход:
        image  : np.ndarray, uint8, [H, W]
        method : "global" | "otsu" | "adaptive_mean" | "adaptive_gaussian"
        params : t, block_size, C и т. п.
    Выход:
        np.ndarray, bool, [H, W]; True — объект (штрих/монета), False — фон.
    Замечание: следите за полярностью. Тёмный объект на светлом фоне требует
    инвертирующего порога; несогласованная полярность обесценивает метрики.
    '''
    raise NotImplementedError


# TODO: серия по методам и параметрам, запись в журнал через
# log_run(stage="binarization", ...), визуальное сравнение на uneven_page и page.

## 10. Задание 3. Морфологическая очистка

Морфология применяется к результату бинаризации, а не к полутоновому изображению (полутоновая морфология существует, но в задании речь о бинарной).

Соответствие операций дефектам: **открытие** удаляет одиночные светлые точки («соль») внутри фона, **закрытие** заполняет тёмные точки («перец») внутри объекта. Применение закрытия против «соли» и открытия против «перца» — характерная ошибка, которая видна в отчёте сразу.

Отдельно проследите, как размер и форма структурного элемента (`cv2.MORPH_RECT`, `MORPH_ELLIPSE`, `MORPH_CROSS`) влияют на сохранность тонких штрихов: слишком большое ядро при открытии разрушает сам текст.

In [ ]:
# Рабочий пример (рельсы): бинарная маска с импульсным дефектом и открытие.
noisy_page = add_salt_pepper(IMAGES["uneven_page"], amount=0.03, seed=SEED)
_, binary = cv2.threshold(noisy_page, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

for method, mask in [("none", binary), ("open_rect3", opened)]:
    log_run(image="uneven_page", stage="morphology", method=method,
            params={"kernel": "rect3"}, **pixel_prf(page_gt, mask > 0))

show_row([noisy_page, binary, opened], ["вход с шумом", "бинаризация", "открытие 3x3"])
runs_table("morphology")

In [ ]:
# TODO (задание 3.2): исследуйте все четыре морфологические операции.
#
# Требования:
# 1. Эрозия, дилатация, открытие, закрытие — каждая применена и объяснена.
# 2. Не менее двух размеров структурного элемента и не менее двух его форм.
# 3. Показано, какая операция соответствует какому типу дефекта («соль» / «перец»),
#    с количественным подтверждением (precision/recall/F1 из pixel_prf).
# 4. Показан режим переочистки: ядро, при котором полезная структура разрушается.

def morphology_clean(binary: np.ndarray, operation: str, ksize: int = 3,
                     shape: str = "rect", iterations: int = 1) -> np.ndarray:
    '''Морфологическая очистка бинарной маски.

    Вход:
        binary    : np.ndarray, bool [H, W]
        operation : "erode" | "dilate" | "open" | "close"
        ksize     : размер структурного элемента (нечётный)
        shape     : "rect" | "ellipse" | "cross"
    Выход:
        np.ndarray, bool [H, W].
    '''
    raise NotImplementedError


# TODO: серия и запись в журнал со stage="morphology".

## Отчёт

Заполните три сводные таблицы (код ниже строит их из журнала) и напишите выводы.

**Таблица 1 — обязательная по заданию:** «тип шума × фильтр → PSNR/SSIM». Строка `none` — baseline.

**Таблица 2:** бинаризация — метод × метрики на изображении с неравномерным освещением.

**Таблица 3:** морфология — операция и структурный элемент × метрики.

Требования к выводам (см. [общие МУ](../../../docs/guidelines-students.md), пп. 2–3):

- **Наблюдение** — измеренный факт: «на camera при шуме соль-перец p=0.05 медианный фильтр 3×3 дал PSNR 30.1 дБ против 21.4 дБ у усредняющего 3×3».
- **Интерпретация** — предполагаемая причина: «медиана робастна к выбросам, усреднение размазывает импульс по окрестности».
- **Вывод с указанием границ** — «в проверенном диапазоне p ≤ 0.05 и ядер 3–5 для импульсного шума предпочтителен медианный фильтр; для гауссова шума при σ ≥ 20 преимущество не подтверждено».

Утверждения вида «медианный фильтр лучше» без указания типа шума, параметров и метрики выводом не считаются.

In [ ]:
# Сводные таблицы из журнала экспериментов.
filtering = runs_table("filtering")

if not filtering.empty:
    table1 = filtering.pivot_table(index=["noise", "method"], values=["psnr", "ssim", "ms"],
                                   aggfunc="mean").round(3)
    display(table1)
else:
    print("Журнал фильтрации пуст: выполните задания 2.1-2.3.")

for stage in ("binarization", "morphology"):
    df = runs_table(stage)
    if not df.empty:
        print(f"\n{stage}:")
        display(df.groupby(["image", "method"])[["precision", "recall", "f1"]].mean().round(3))

# TODO: при необходимости добавьте графики (PSNR по размеру ядра, F1 по размеру
# структурного элемента). Не смешивайте на одном графике несопоставимые конфигурации.

### Выводы

Заполните, опираясь на таблицы выше. Каждый пункт — с указанием изображения, параметров и метрики.

**Наблюдения**

1.
2.
3.

**Интерпретация**

1.
2.

**Выводы и границы применимости**

1.
2.

**Анализ характерных ошибок.** Приведите не менее двух случаев, где метод сработал плохо (например, размытие текстуры гауссовым фильтром, разрушение тонких штрихов открытием, провал Оцу на неравномерном освещении), с иллюстрацией и объяснением причины.

**Использование сторонних материалов и LLM.** Укажите источники и характер использования (п. 5 общих МУ).

## Контрольные вопросы

Из [списка вопросов блока](README.md#контрольные-вопросы-блока), относящиеся к этой работе. Ответы включите в отчёт; на защите они задаются устно.

1. Чем отличаются цветовые пространства RGB, HSV и Lab и когда какое использовать?
2. Почему медианный фильтр лучше усредняющего подавляет шум «соль-перец»?
3. Что такое частотная фильтрация и как связаны свёртка и произведение спектров?
4. В чём разница между эрозией, дилатацией, открытием и закрытием?

Дополнительно к защите: почему для выбранного типа шума вы выбрали именно этот фильтр и что показывает частотная область?

## Чек-лист перед сдачей

Полный список — в [общих МУ, п. 6](../../../docs/guidelines-students.md#6-чек-лист-перед-сдачей). Специфика ДЗ1:

- [ ] Ноутбук исполняется сверху вниз без ошибок после `Restart & Run All`.
- [ ] Указаны ФИО, группа, версии библиотек и seed.
- [ ] В наборе не менее 5 изображений, есть цветные и полутоновые.
- [ ] Реализованы 2 типа шума и не менее 3 фильтров, есть baseline `none`.
- [ ] PSNR и SSIM посчитаны относительно эталона, а не относительно зашумлённого входа.
- [ ] Есть частотная фильтрация через БПФ и сравнение с пространственной.
- [ ] Оцу и адаптивный порог сравнены на изображении с неравномерным освещением.
- [ ] Применены все четыре морфологические операции, соответствие «операция → дефект» обосновано.
- [ ] Есть сводная таблица «шум × фильтр → PSNR/SSIM» и визуальные серии «до/после».
- [ ] Наблюдения отделены от интерпретаций, границы выводов указаны.
- [ ] Разобраны не менее двух неудачных случаев.